[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.2 MB/s eta 0:00:00


In [2]:
import torch

In [25]:
# ✏️ YOUR IMPLEMENTATION HERE
def my_softmax(logits):
    e = torch.exp(logits - logits.max())
    es = torch.sum(e, dim=-1, keepdim=True)
    return e / es


def my_cross_entropy_loss(logits, targets):
    losses = []
    for t in range(len(targets)):
        q = my_softmax(logits[t])
        losses.append(-torch.log(q[targets[t]]))
    return torch.mean(torch.stack(losses))

def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
  # logits: (B, C), targets: (B,)
  B = logits.size(0)

  # 1) 取出每个样本对应正确类别的 logit, x_y     → shape (B,)
  x_y = logits[torch.arange(B), targets]

  # 2) 沿类别维度做 logsumexp, 内部已经做了减 max → shape (B,)
  lse = torch.logsumexp(logits, dim=-1)

  # 3) 每个样本 CE = lse - x_y, 再对 batch 平均  → scalar
  return (lse - x_y).mean()

In [23]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(2.4274)
Ref:  tensor(2.4274)


In [26]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (7.9ms)
  ✅ [2/4] Numerical stability (0.7ms)
  ✅ [3/4] Scalar output (0.4ms)
  ✅ [4/4] Gradient flow (4.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (13.2ms total)
  Progress saved. Run status() to see your dashboard.



In [11]:
from torch_judge import check, status, hint, reset_progress
hint("cross_entropy")


💡 Hint for Cross-Entropy Loss:
   log_probs = logits - logsumexp(logits, dim=-1, keepdim=True). Loss = -log_probs[arange(B), targets].mean(). Subtract max for stability (logsumexp handles this).

